# 🔎 NeuralShield v6.3 — Hata Analizi + Sızıntı + MT-Çeşitlilik Kapısı

Veri üretmeden **önce** (plandaki §4 kapısı):
1. mmBERT v6.2'nin adil benchmark'taki **FN/FP**'lerini dile, uzunluğa, kaynağa göre dök.
2. `heldout_benchmark_v6_fair.json` ↔ `train_v6_clean.parquet` **tam + yakın-kopya** sızıntı kontrolü.
3. **MT-tohum çeşitliliği**: 34k MT saldırı kaç benzersiz kümeye düşüyor.

**T4 yeterli, ~10-15 dk.** Çıktı: A/B/C kararını belirler.

In [ ]:
# @title 1. Kurulum + Drive + Repo
import os, sys, json, re, random, unicodedata, gc, time
import warnings; warnings.filterwarnings('ignore')
os.environ['TOKENIZERS_PARALLELISM']='false'
try:
    from google.colab import drive; drive.mount('/content/drive')
except Exception as e: print('drive:', e)
!pip -q install -U transformers datasets datasketch scikit-learn 2>/dev/null
if not os.path.exists('ai-soc-agent-lab'):
    !git clone -q https://github.com/Mustafa-Caliskan/ai-soc-agent-lab.git
if os.path.exists('ai-soc-agent-lab'): os.chdir('ai-soc-agent-lab')
!git pull -q
import numpy as np, torch, pandas as pd
MASTER='/content/drive/MyDrive/NeuralShield_v6_MasterData'
MM='/content/drive/MyDrive/NeuralShield_v6_mmbert'
FAIR=MASTER+'/heldout_benchmark_v6_fair.json'
PARQUET=MASTER+'/train_v6_clean.parquet'
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('fair var:', os.path.exists(FAIR), '| parquet var:', os.path.exists(PARQUET))

In [ ]:
# @title 2. mmBERT tahminleri + FN/FP dökümü (dil / uzunluk / kaynak)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
def best_ckpt(base):
    if os.path.exists(base+'/config.json'): return base
    if os.path.isdir(base+'/checkpoints'):
        subs=[d for d in os.listdir(base+'/checkpoints') if d.startswith('checkpoint-')]
        if subs: return os.path.join(base+'/checkpoints', sorted(subs, key=lambda x:int(x.split('-')[1]))[-1])
    return None
ck=best_ckpt(MM); print('checkpoint:', ck)
tok=AutoTokenizer.from_pretrained(ck); mod=AutoModelForSequenceClassification.from_pretrained(ck, torch_dtype=torch.float16).to('cuda').eval()
data=json.load(open(FAIR, encoding='utf-8'))
texts=[x['text'] for x in data]; labels=np.array([int(x['label']) for x in data]); probs=[]
for i in range(0,len(texts),64):
    inp=tok(texts[i:i+64], return_tensors='pt', padding=True, truncation=True, max_length=512).to('cuda')
    with torch.no_grad(): probs.extend(torch.softmax(mod(**inp).logits.float(),-1)[:,1].cpu().numpy().tolist())
probs=np.array(probs); pr=(probs>=0.70).astype(int)
del mod,tok; gc.collect(); torch.cuda.empty_cache()
print('GENEL @0.7: Recall=%.1f FP=%.1f Acc=%.1f' % (
    100*pr[labels==1].mean(), 100*pr[labels==0].mean(), 100*(pr==labels).mean()))

df=pd.DataFrame({'text':texts,'label':labels,'pred':pr,'prob':probs,
                 'lang':[x['language'] for x in data],'source':[x['source'] for x in data],
                 'length':[len(t) for t in texts]})
print('\n--- DİL BAZINDA (recall@1 / fp@0) ---')
for lg in ['en','tr','de','fr','es','it','pt','ru','ar','ja','ko','zh']:
    d=df[df.lang==lg]
    a=d[d.label==1]; b=d[d.label==0]
    print(f'  {lg}: rec={100*a.pred.mean():.1f} fp={100*b.pred.mean():.1f} (n={len(d)})')

print('\n--- UZUNLUK HİPOTEZİ (FN=kaçırılan saldırı, FP=yanlış alarm) ---')
bins=[0,150,300,600,1200,10000]
df['lbin']=pd.cut(df.length,bins)
fn=df[(df.label==1)&(df.pred==0)]; fp=df[(df.label==0)&(df.pred==1)]
print('  FN uzunluk medyanı:', fn.length.median(), '| TP medyanı:', df[(df.label==1)&(df.pred==1)].length.median())
print('  FN uzunluk dağılımı:', fn.lbin.value_counts().sort_index().to_dict())
print('  FP uzunluk dağılımı:', fp.lbin.value_counts().sort_index().to_dict())
print('  >600 karakter saldırı sayısı:', int(((df.label==1)&(df.length>600)).sum()), '| bunlardan kaçan:', int(((df.label==1)&(df.length>600)&(df.pred==0)).sum()))

print('\n--- KAYNAK BAZINDA (saldırı recall) ---')
for s,g in df[df.label==1].groupby('source'):
    if len(g)>=10: print(f'  {s[:45]}: rec={100*g.pred.mean():.1f} (n={len(g)})')
df[['lang','label','pred','prob','length','source']].to_csv(MASTER+'/error_analysis_predictions.csv', index=False)
print('\n-> predictions kaydedildi: error_analysis_predictions.csv')

In [ ]:
# @title 3. SIZINTI: fair benchmark ↔ train parquet (tam + yakın-kopya MinHash)
from datasketch import MinHash, MinHashLSH
def norm(s):
    s=str(s or '')
    s=unicodedata.normalize('NFKC',s).lower()
    s=re.sub(r'\s+',' ',s).strip()
    return s
def shingles(s,k=5):
    s=norm(s); return set(s[i:i+k] for i in range(max(0,len(s)-k+1)))
def mh(s):
    m=MinHash(num_perm=64)
    for sh in shingles(s): m.update(sh.encode('utf-8'))
    return m
fair=[norm(x['text']) for x in data]
fair_set=set(fair)
train=pd.read_parquet(PARQUET)
tr_texts=[str(t) for t in train['text'].tolist()]
tr_norm=[norm(t) for t in tr_texts]
tr_set=set(tr_norm)
exact=sum(1 for t in fair if t in tr_set)
print(f'FAIR n={len(fair)} | TRAIN n={len(tr_norm)}')
print(f'TAM eşleşme (normalize): {exact} / {len(fair)}')
# near-dup
t0=time.time(); lsh=MinHashLSH(threshold=0.85, num_perm=64)
for i,t in enumerate(tr_norm):
    try: lsh.insert(str(i), mh(t))
    except Exception: pass
print(f'LSH kuruldu ({time.time()-t0:.1f}s)')
near=0; near_examples=[]
for i,(orig,t) in enumerate(zip(texts,fair)):
    try:
        res=lsh.query(mh(t))
        if res:
            near+=1
            if len(near_examples)<10: near_examples.append((orig[:80], tr_texts[int(res[0])][:80]))
    except Exception: pass
print(f'YAKIN-KOPYA (Jaccard>=0.85): {near} / {len(fair)}  ({100*near/len(fair):.1f}%)')
for a,b in near_examples: print('   FAIR:',a,'\n   TRAIN:',b)

In [ ]:
# @title 4. MT-TOHUM ÇEŞİTLİLİĞİ (supplement MT saldırıları kaç benzersiz kümeye düşüyor)
from collections import Counter
SUP='data/v6_multilingual_supplement.json'
sup=json.load(open(SUP,encoding='utf-8')) if os.path.exists(SUP) else []
print('supplement N:', len(sup))
for lg in ['tr','ru','de','fr','es','it','pt','ar','ja','ko','zh']:
    xs=[str(x['text']) for x in sup if x.get('label')==1 and x.get('language')==lg and (x.get('src')=='mt' or x.get('tier')==2)]
    if not xs: continue
    lsh2=MinHashLSH(threshold=0.85,num_perm=32); uniq=0
    for i,t in enumerate(xs):
        mm=MinHash(num_perm=32)
        for sh in shingles(t):
            mm.update(sh.encode('utf-8'))
        if not lsh2.query(mm):
            lsh2.insert(str(i), mm); uniq+=1
    print(f'  {lg}: MT atk={len(xs)} | benzersiz küme≈{uniq} | oran={100*uniq/len(xs):.1f}%')
print('\nYorum: benzersiz küme << toplam ise MT çeşitliliği düşüktür; yeni çeviri yerine insan-yazımı veri gerekir.')